# Dataset Download Link

https://www.kaggle.com/datasets/jypenpen54534/steam-games-dataset-2021-2025-65k

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/jamessalcedo/steam-data/a_steam_data_2021_2025.csv')

# Checking the initial state of the data
print("Original row count:", len(df))
print("Missing values before cleaning:\n", df.isnull().sum())

# Cleaning data
df = df.dropna(subset=['genres', 'developer', 'publisher'])

# Transforming data
df['genres_list'] = df['genres'].str.split(';')

# Checking cleaned data
print("\nCleaned row count:", len(df))
display(df.head(3))

In [ ]:
# 1. Compare Free vs. Paid Games
df['revenue_model'] = df['price'].apply(lambda x: 'Free' if x == 0.0 else 'Paid')

pricing_stats = df.groupby('revenue_model').agg(
    total_games=('appid', 'count'),
    avg_recommendations=('recommendations', 'mean')
).round(2)

print("--- Free vs. Paid Engagement ---")
display(pricing_stats)

# 2. Top 10 Developers by Player Reception
top_developers = df.groupby('developer').agg(
    total_recommendations=('recommendations', 'sum'),
    games_released=('appid', 'count')
).sort_values(by='total_recommendations', ascending=False).head(10)

print("\n--- Top 10 Developers ---")
display(top_developers)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_devs_clean = top_developers.reset_index()
top_devs_clean['developer'] = top_devs_clean['developer'].apply(lambda x: (x[:25] + '...') if len(x) > 25 else x)

sns.set_theme(style="darkgrid")
plt.figure(figsize=(16, 7))

# Chart 1: Top 10 Developers by Recommendations
plt.subplot(1, 2, 1)
sns.barplot(
    data=top_devs_clean, 
    y='developer', 
    x='total_recommendations', 
    hue='developer',
    palette='viridis',
    legend=False
)
plt.title('Top 10 Developers by Total Recommendations', fontsize=14, pad=15)
plt.xlabel('Total Recommendations (Millions)')
plt.ylabel('Developer')

# Chart 2: Free vs. Paid Average Recommendations
plt.subplot(1, 2, 2)
sns.barplot(
    data=pricing_stats.reset_index(), 
    x='revenue_model', 
    y='avg_recommendations', 
    hue='revenue_model',
    palette='pastel',
    legend=False
)
plt.title('Average Recommendations: Free vs. Paid', fontsize=14, pad=15)
plt.xlabel('Revenue Model')
plt.ylabel('Average Recommendations')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
yearly_counts = df['release_year'].value_counts().sort_index()

sns.lineplot(x=yearly_counts.index, y=yearly_counts.values, marker='o', color='purple', linewidth=2.5)
plt.title('Number of Games Released per Year (2021-2025)', fontsize=14, pad=15)
plt.xlabel('Release Year')
plt.ylabel('Total Games Released')
plt.xticks(yearly_counts.index) 
plt.show()

In [ ]:
categories_expanded = df.assign(category=df['categories'].str.split(';')).explode('category')
top_categories = categories_expanded['category'].value_counts().head(10).reset_index()
top_categories.columns = ['category', 'count']

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, y='category', x='count', hue='category', palette='magma', legend=False)
plt.title('Top 10 Most Common Game Features', fontsize=14, pad=15)
plt.xlabel('Number of Games')
plt.ylabel('Feature / Category')
plt.tight_layout()
plt.show()

In [ ]:
# Get the top 5 genres by volume
df_genres = df.assign(genre=df['genres'].str.split(';')).explode('genre')
top_5_genres = df_genres['genre'].value_counts().head(5).index

# Filter the data to only include those 5 genres and games priced $60 or under
price_data = df_genres[df_genres['genre'].isin(top_5_genres) & (df_genres['price'] <= 60)]

plt.figure(figsize=(12, 6))
sns.boxplot(data=price_data, x='genre', y='price', hue='genre', palette='Set2', legend=False)
plt.title('Price Distribution Across Top 5 Genres (Capped at $60)', fontsize=14, pad=15)
plt.xlabel('Genre')
plt.ylabel('Price ($)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

scatter_data = df[(df['price'] <= 60) & (df['recommendations'] <= 50000)]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=scatter_data, 
    x='price', 
    y='recommendations', 
    alpha=0.4, 
    color='teal'
)
plt.title('Game Price vs. Total Recommendations', fontsize=14, pad=15)
plt.xlabel('Price ($)')
plt.ylabel('Recommendations')
plt.tight_layout()
plt.show()

In [ ]:
top_publishers = df['publisher'].value_counts().head(10).reset_index()
top_publishers.columns = ['publisher', 'game_count']

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_publishers, 
    y='publisher', 
    x='game_count', 
    hue='publisher',
    palette='crest', 
    legend=False
)
plt.title('Top 10 Publishers by Total Game Releases', fontsize=14, pad=15)
plt.xlabel('Number of Games Published')
plt.ylabel('Publisher')
plt.tight_layout()
plt.show()

# **Recommendation Predictor**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df['is_free'] = (df['price'] == 0).astype(int)

X = df[['price', 'release_year', 'is_free']]
y = df['recommendations']

# Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Prediction and Evaluation
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (Recommendations): {mae:.2f}")
print(f"R-squared Score: {r2:.4f}")

A Random Forest model was trained to predict a game's player recommendations based solely on its price and release year. The resulting R-squared score of 0.015 indicates a near-zero correlation. This proves a critical business insight: pricing strategy alone does not dictate player engagement. 

A game cannot guarantee a specific volume of player reception simply by pricing itself at an 'optimal' tier. Success is driven heavily by qualitative factors (gameplay, graphics, marketing) rather than basic metadata.